# Match articles and supporting information

Match DOI-based filenames to the acquisition inventory and create the manifest used by text mining. Matching retains row order, duplicate DOI rows, and the configured SI extension preference. See [document matching](../docs/document_matching.md) for inputs and counting conventions.

## Inputs and setup

Install `python -m pip install -e ".[mining,notebook]"` from the repository root. Run the cells in order. The default configuration matches the included illustrative article/SI pair locally and needs no API key. All paths below are repository-relative.

| Input | Included example | For a collection of real papers |
| --- | --- | --- |
| DOI inventory, CSV | `Demo/03_api_demo/inputs/mining/inventory.csv` | A CSV or XLSX containing `DOI`; set `input_file` in `configs/document_matching.json`. |
| Main articles, PDF | `Demo/03_api_demo/inputs/mining/articles/` | Put articles in `data/local/articles/`. |
| Supporting information, PDF or DOCX | `Demo/03_api_demo/inputs/mining/supporting_information/` | Put SI files in `data/local/supporting_information/`. |

Use `10.1021_jacs.2c09756.pdf` and `10.1021_jacs.2c09756_SI.pdf` as filename patterns: replace the DOI slash with an underscore and add `_SI` for supporting information. For real papers, select `configs/document_matching.json` in the configuration cell. Its default DOI inventory is `data/processed_data/literature_retrieval/supporting_information.csv`. The included PDF text is illustrative; replace it with the corresponding real documents for literature extraction.

Matching writes the manifest automatically. The example manifest is `results/examples/mining/document_manifest.csv`; the main workflow writes `results/extraction/document_manifest.csv`. Optional counts require `.[document-counts]` and `RUN_COUNTS = True`.

Implementation: [document matching and counts](../src/mofinder/literature/match_documents.py). A [three-paper input template](../Demo/03_api_demo/README.md) is also available. See the [source-to-code guide](../docs/source_to_code.md) for the original workflow stages and their corresponding functions.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError("Run this notebook from the repository root or notebooks directory.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from mofinder.display import display_paths
from mofinder.literature.match_documents import load_config, run_matching, run_counting, read_table


## Configuration

The example configuration uses the illustrative article and SI pair included in the repository. Set `CONFIG_FILE` to `configs/document_matching.json` to match the acquisition inventory against `data/local/`.


In [ ]:
CONFIG_FILE = PROJECT_ROOT / "configs/example_document_matching.json"
config = load_config(CONFIG_FILE)


In [ ]:
matching_summary = run_matching(config)
matching_summary


In [ ]:
manifest = read_table(config["manifest_file"])
manifest.head().apply(lambda column: column.map(display_paths))


## Optional counts

Counting does not call a model. It extracts PDF text and uses the configured tokenizer; tokenizer assets may be downloaded on first use. `Combined Words` and `Combined Tokens` prefer SI counts and fall back to article counts only when the SI count is missing. Unsupported or unreadable existing files receive zero counts and are listed in the summary. No OCR or DOC/DOCX text extraction is performed.


In [ ]:
RUN_COUNTS = False
SAVE_PLOTS = False

if RUN_COUNTS:
    count_summary = run_counting(config, plots=SAVE_PLOTS)
    display(display_paths(count_summary))
